In [3]:
import numpy as np
import pandas as pd
import re
import string
import nltk
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [4]:
df = pd.read_csv("/content/WELFake_Dataset.csv",
                  engine='python',
                  on_bad_lines='skip')

In [5]:
df.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [6]:
df.columns = df.columns.str.strip()
print("Columns:", df.columns.tolist())

Columns: ['Unnamed: 0', 'title', 'text', 'label']


In [7]:
df.shape

(71924, 4)

In [8]:
df.info()






<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71924 entries, 0 to 71923
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  71924 non-null  object
 1   title       71359 non-null  object
 2   text        71867 non-null  object
 3   label       71904 non-null  object
dtypes: object(4)
memory usage: 2.2+ MB


In [9]:
df.describe()

,Unnamed: 0,title,text,label
count,71924,71359,71867,71904
unique,71924,62179,62539,4
top,71903,Factbox: Trump fills top jobs for his administ...,,1
freq,1,14,734,36984


In [10]:
df.isnull().sum()

,0
Unnamed: 0,0
title,565
text,57
label,20


In [11]:
df['text'] = df['title'].fillna('') + " " + df['text'].fillna('')
df = df[['text', 'label']]

In [12]:
df.head()

,text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,1
1,Did they post their votes for Hillary already?,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",0
4,SATAN 2: Russia unvelis an image of its terrif...,1


In [13]:
df = df[df['label'].isin([0, 1, '0', '1'])].copy()
df['label'] = df['label'].astype(int)

In [14]:
df.head()

,text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,1
1,Did they post their votes for Hillary already?,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",0
4,SATAN 2: Russia unvelis an image of its terrif...,1


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 71902 entries, 0 to 71923
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    71902 non-null  object
 1   label   71902 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 1.6+ MB


In [16]:
df.isnull().sum()

,0
text,0
label,0


**Text** **Cleaning**

In [17]:
noise_words = [
    'said', 'would', 'could', 'also', 'one',
    'us', 'new', 'like', 'get', 'make',
    'time', 'year', 'day',
    'news', 'report', 'according',
    'reuters', 'ap', 'cnn', 'krtv'
]
def clean_text(text):
    #Handle null safely
    if pd.isnull(text):
        return ""


    text = text.lower()#Lowercase


    text = re.sub(r'http\S+|www\S+', '', text) # Remove URLs


    text = re.sub(r'<.*?>', '', text) # Remove HTML tags


    text = re.sub(r'\d+', '', text) #Remove numbers


    text = text.translate(str.maketrans('', '', string.punctuation)) #Remove punctuation


    text = re.sub(r'\s+', ' ', text).strip()#Remove extra spaces


    tokens = word_tokenize(text) #Tokenization

    #  Remove stopwords + lemmatization + filter small words
    """tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words and len(word) > 2
    ]"""
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
        and word not in noise_words
        and len(word) > 2
    ]


    return " ".join(tokens)
df['text']=df['text'].apply(clean_text)


In [18]:
df['text']

,text
0,law enforcement high alert following threat co...
1,post vote hillary already
2,unbelievable obama attorney general say charlo...
3,bobby jindal raised hindu us story christian c...
4,satan russia unvelis image terrifying supernuk...
...,...
71919,megyn kelly jump nbc fox test network york tim...
71920,heck gore meeting trump team today video talk ...
71921,obama announces lifting sanction myanmar washi...
71922,forbes list world powerful people out…and obam...


In [19]:
df['text'][8]

'sport bar owner ban nfl games…will show true american sport speak rural america video owner ringling bar located south white sulphur spring standing behind facebook post criticizes nfl player take knee national anthem protest police brutality post made ringling bar facebook page tuesday night since received hundred comment share post read ringling bar longer show nfl game allow air pbr rodeo nascar event whose competitor true american sorry inconvenience ringling bar coowner kurt bekemans grew paradise valley published post care post turn customer away seriously care nonamericans patronize place bekemans speak rural america bet see farmer rancher whole country take knee guy bet find appreciate great nation given think least give thanks country stand flag anthem wednesday morning majority comment support bar love many people round commenteranother person critical writing course nascar protesting treatment minority care read'

**Text and padding**

In [23]:
max_words = 20000
max_length = 300



X_train, X_test, y_train, y_test = train_test_split(
df['text'], df['label'], test_size=0.2, random_state=42)

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)  # SIRF TRAIN PE

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train = pad_sequences(X_train_seq, maxlen=max_length)
X_test = pad_sequences(X_test_seq, maxlen=max_length)

In [25]:
X_train

array([[   0,    0,    0, ...,  382, 1139,   59],
       [  81,  158,    1, ...,  149,  510, 3347],
       [   0,    0,    0, ...,  197, 1396, 1663],
       ...,
       [   0,    0,    0, ..., 1488,  262, 2673],
       [   0,    0,    0, ..., 1385,  756,  800],
       [   0,    0,    0, ...,   25,  272, 5813]], dtype=int32)

In [26]:
y_train.shape

(57521,)

In [ ]:
df['label'].value_counts()

In [27]:
model = Sequential([
    Embedding(20000, 128, input_length=300),
    LSTM(64, return_sequences=True),
    Dropout(0.5),
    LSTM(32),
    Dropout(0.4),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [28]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=5,
    batch_size=32,
    class_weight=class_weight_dict,  # YEH ADD KARO
    verbose=1
)


Class weights: {0: np.float64(1.0319148936170213), 1: np.float64(0.97)}
Epoch 1/5
1798/1798 ━━━━━━━━━━━━━━━━━━━━ 58s 28ms/step - accuracy: 0.8926 - loss: 0.2642 - val_accuracy: 0.9396 - val_loss: 0.1669
Epoch 2/5
1798/1798 ━━━━━━━━━━━━━━━━━━━━ 51s 28ms/step - accuracy: 0.9503 - loss: 0.1358 - val_accuracy: 0.9515 - val_loss: 0.1350
Epoch 3/5
1798/1798 ━━━━━━━━━━━━━━━━━━━━ 49s 27ms/step - accuracy: 0.9746 - loss: 0.0733 - val_accuracy: 0.9652 - val_loss: 0.1039
Epoch 4/5
1798/1798 ━━━━━━━━━━━━━━━━━━━━ 50s 28ms/step - accuracy: 0.9863 - loss: 0.0425 - val_accuracy: 0.9606 - val_loss: 0.1474
Epoch 5/5
1798/1798 ━━━━━━━━━━━━━━━━━━━━ 49s 27ms/step - accuracy: 0.9935 - loss: 0.0210 - val_accuracy: 0.9672 - val_loss: 0.1555


In [29]:
y_pred = (model.predict(X_test) > 0.5).astype("int32")
loss, acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", acc)

450/450 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step
450/450 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9672 - loss: 0.1555
Test Accuracy: 0.9671789407730103


In [30]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.96      0.97      0.97      7047
           1       0.97      0.96      0.97      7334

    accuracy                           0.97     14381
   macro avg       0.97      0.97      0.97     14381
weighted avg       0.97      0.97      0.97     14381



In [39]:
def predict_news(news):
    news = clean_text(news)
    seq = tokenizer.texts_to_sequences([news])
    padded = pad_sequences(seq, maxlen=300)
    pred = model.predict(padded)[0][0]

    print("Confidence:", pred)

    if pred > 0.5:
        return "Real News "
    else:
        return "Fake News "

In [40]:
print(predict_news("Government announces new education policy"))
print(predict_news("Regional situation to be discussed in meeting with Saudi crown prince; premier to attend Antalya Diplomacy Forum, meet Erdogan in Turkiye."))
print(predict_news('Pakistan has secretly banned WhatsApp and replaced it with a new government app called ‘PakChat’ starting from next week.')
)
print(predict_news("Government announces new education policy"))
print(predict_news("Lawyers demand provision of B-class facilities to the couple in jail; call for an end to harassment lawyers over social media statements."))

print(predict_news("Pakistan government has announced complete internet shutdown for one year starting next month"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Confidence: 0.99914396
Real News 
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Confidence: 0.9830488
Real News 
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
Confidence: 0.026348403
Fake News 
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
Confidence: 0.99914396
Real News 
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
Confidence: 0.95084375
Real News 
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
Confidence: 0.2994477
Fake News 
